# Project 11 -- Conformal Risk Prediction

**Distribution-free prediction intervals for financial risk measures with finite-sample coverage guarantees.**

This notebook walks through the core techniques of conformal prediction applied to financial risk:

1. **Section 1** -- Generate synthetic data and split into train/cal/test
2. **Section 2** -- Split conformal prediction: compute intervals, verify coverage
3. **Section 3** -- CQR for VaR intervals: symmetric vs asymmetric comparison
4. **Section 4** -- Conformal PD intervals: ML uncertainty for credit scoring
5. **Section 5** -- ACI experiment: regime change and coverage adaptation
6. **Section 6** -- Method comparison: conformal vs parametric vs bootstrap

**Key insight:** Conformal prediction provides *distribution-free* coverage guarantees -- no normality assumption, no asymptotic regime. The coverage holds in finite samples under exchangeability.

In [ ]:
import sys
from pathlib import Path

# Add project and shared library to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root.parent.parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from risk_analyst.models.conformal import (
    split_conformal_threshold,
    conformal_prediction_interval,
    cqr_threshold,
    cqr_interval,
    conformal_risk_control,
    adaptive_conformal_update,
)
from risk_analyst.measures.var import historical_var, parametric_var

from models import QuantileRegressor, ConformalVaR, ConformalPD
from adaptive import AdaptiveConformalInference, generate_regime_data, run_aci_experiment
from model import ConformalRiskModel
from diagnostics import (
    plot_coverage_over_time,
    plot_interval_width_comparison,
    plot_adaptive_coverage,
    plot_conformal_pd,
)

# Load config
with open(project_root / "configs" / "default.yaml") as f:
    config = yaml.safe_load(f)

plt.style.use("seaborn-v0_8-whitegrid")
SEED = config["random_seed"]
rng = np.random.default_rng(SEED)

print(f"Config loaded. Random seed: {SEED}")
print(f"Target coverage: {1 - config['conformal']['alpha']:.0%}")

## Section 1: Generate Synthetic Data

We create two datasets:
- **Regression data** (for VaR intervals): linear model with Gaussian noise, representing return prediction
- **Credit data** (for PD intervals): logistic model with binary default labels

Both are split into train / calibration / test sets. The calibration set is the key ingredient for conformal prediction -- it provides the nonconformity scores used to set the prediction interval width.

In [ ]:
# --- Regression data (for VaR / return prediction) ---
n_reg = config["data"]["n_samples"]
n_features = config["data"]["n_features"]
X_reg = rng.standard_normal((n_reg, n_features))
beta_true = np.array([1.0, -0.5, 0.3, 0.0, 0.2])
y_reg = X_reg @ beta_true + rng.standard_normal(n_reg) * 0.5

# Split: 50% train, 20% calibration, 30% test
n_train = int(n_reg * 0.5)
n_cal = int(n_reg * 0.2)
X_train, y_train = X_reg[:n_train], y_reg[:n_train]
X_cal, y_cal = X_reg[n_train:n_train + n_cal], y_reg[n_train:n_train + n_cal]
X_test, y_test = X_reg[n_train + n_cal:], y_reg[n_train + n_cal:]

print(f"Regression data: {n_reg} samples, {n_features} features")
print(f"  Train: {len(X_train)}, Calibration: {len(X_cal)}, Test: {len(X_test)}")

# --- Credit data (for PD intervals) ---
n_credit = config["credit"]["n_samples"]
X_credit = rng.standard_normal((n_credit, n_features))
logits = X_credit @ np.array([0.5, -0.3, 0.2, 0.1, -0.4]) - 1.5
probs_true = 1 / (1 + np.exp(-logits))
y_credit = rng.binomial(1, probs_true)

n_cr_train = int(n_credit * 0.5)
n_cr_cal = int(n_credit * 0.2)
X_cr_train, y_cr_train = X_credit[:n_cr_train], y_credit[:n_cr_train]
X_cr_cal, y_cr_cal = X_credit[n_cr_train:n_cr_train + n_cr_cal], y_credit[n_cr_train:n_cr_train + n_cr_cal]
X_cr_test, y_cr_test = X_credit[n_cr_train + n_cr_cal:], y_credit[n_cr_train + n_cr_cal:]

print(f"\nCredit data: {n_credit} samples, default rate: {y_credit.mean():.2%}")
print(f"  Train: {len(X_cr_train)}, Calibration: {len(X_cr_cal)}, Test: {len(X_cr_test)}")

## Section 2: Split Conformal Prediction

The simplest conformal method: fit any model on training data, compute residuals on the calibration set, then use the quantile of those residuals as the interval half-width.

**Key guarantee:** If (X_1, Y_1), ..., (X_{n+1}, Y_{n+1}) are exchangeable, then

$$\mathbb{P}(Y_{n+1} \in C(X_{n+1})) \geq 1 - \alpha$$

where the threshold is set as the $\lceil (1-\alpha)(1+1/n) \rceil$-th quantile of the calibration scores.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

alpha = config["conformal"]["alpha"]

# Step 1: Fit a model on training data
model = GradientBoostingRegressor(
    n_estimators=config["cqr"]["n_estimators"],
    max_depth=config["cqr"]["max_depth"],
    random_state=SEED,
)
model.fit(X_train, y_train)

# Step 2: Compute nonconformity scores on calibration set
preds_cal = model.predict(X_cal)
scores_cal = np.abs(y_cal - preds_cal)

# Step 3: Compute the conformal threshold
q_hat = split_conformal_threshold(scores_cal, alpha)
print(f"Conformal threshold (alpha={alpha}): {q_hat:.4f}")
print(f"Calibration score stats: mean={scores_cal.mean():.4f}, "
      f"median={np.median(scores_cal):.4f}, max={scores_cal.max():.4f}")

# Step 4: Construct prediction intervals on test set
preds_test = model.predict(X_test)
lower, upper = conformal_prediction_interval(preds_test, q_hat)

# Step 5: Verify coverage
covered = (y_test >= lower) & (y_test <= upper)
empirical_coverage = covered.mean()
print(f"\nTarget coverage: {1 - alpha:.0%}")
print(f"Empirical coverage: {empirical_coverage:.2%}")
print(f"Average interval width: {(upper - lower).mean():.4f}")

In [ ]:
# Visualize: predictions with conformal bands on a subset of test data
n_show = 80
idx = np.arange(n_show)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(idx, lower[:n_show], upper[:n_show], alpha=0.3, color="steelblue", label="90% conformal interval")
ax.plot(idx, preds_test[:n_show], "b-", linewidth=1, label="Point prediction")
ax.scatter(idx, y_test[:n_show], color="red", s=15, zorder=5, alpha=0.7, label="True value")
ax.set_xlabel("Test observation index")
ax.set_ylabel("Response")
ax.set_title(f"Split Conformal Prediction (coverage = {empirical_coverage:.1%})")
ax.legend()
fig.tight_layout()
plt.show()

## Section 3: Conformalized Quantile Regression (CQR) for VaR Intervals

Split conformal produces **symmetric** intervals (same width above and below the prediction). CQR improves on this by using two quantile regression models -- one for the lower quantile, one for the upper -- producing **asymmetric** intervals that adapt to local data distribution.

The CQR nonconformity score is: $s_i = \max(\hat{q}_{\alpha/2}(X_i) - Y_i, \; Y_i - \hat{q}_{1-\alpha/2}(X_i))$

This measures how far the true value falls outside the predicted quantile interval.

In [ ]:
# Fit lower and upper quantile regressors
lower_qr = QuantileRegressor(
    n_estimators=config["cqr"]["n_estimators"],
    max_depth=config["cqr"]["max_depth"],
    seed=SEED,
)
upper_qr = QuantileRegressor(
    n_estimators=config["cqr"]["n_estimators"],
    max_depth=config["cqr"]["max_depth"],
    seed=SEED + 1,
)

lower_qr.fit(X_train, y_train, quantile=alpha / 2)
upper_qr.fit(X_train, y_train, quantile=1 - alpha / 2)

# Calibrate CQR
lower_cal_preds = lower_qr.predict(X_cal)
upper_cal_preds = upper_qr.predict(X_cal)
q_cqr = cqr_threshold(lower_cal_preds, upper_cal_preds, y_cal, alpha)
print(f"CQR threshold: {q_cqr:.4f}")

# Test intervals
lower_test_preds = lower_qr.predict(X_test)
upper_test_preds = upper_qr.predict(X_test)
cqr_lower, cqr_upper = cqr_interval(lower_test_preds, upper_test_preds, q_cqr)

cqr_covered = (y_test >= cqr_lower) & (y_test <= cqr_upper)
cqr_coverage = cqr_covered.mean()
cqr_widths = cqr_upper - cqr_lower

print(f"\nCQR coverage: {cqr_coverage:.2%} (target: {1-alpha:.0%})")
print(f"CQR avg width: {cqr_widths.mean():.4f}")
print(f"CQR width std: {cqr_widths.std():.4f} (varies across observations)")
print(f"\nCompare with symmetric split conformal:")
print(f"  Split conformal width: {2 * q_hat:.4f} (constant)")
print(f"  CQR avg width:        {cqr_widths.mean():.4f} (adaptive)")

In [ ]:
# Compare symmetric vs asymmetric intervals side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
n_show = 60
idx = np.arange(n_show)

# Symmetric (split conformal)
axes[0].fill_between(idx, lower[:n_show], upper[:n_show], alpha=0.3, color="steelblue")
axes[0].plot(idx, preds_test[:n_show], "b-", linewidth=1)
axes[0].scatter(idx, y_test[:n_show], color="red", s=12, zorder=5, alpha=0.7)
axes[0].set_title(f"Split Conformal (symmetric)\nCoverage={empirical_coverage:.1%}, Width={2*q_hat:.3f}")
axes[0].set_xlabel("Observation")
axes[0].set_ylabel("Response")

# Asymmetric (CQR)
axes[1].fill_between(idx, cqr_lower[:n_show], cqr_upper[:n_show], alpha=0.3, color="coral")
axes[1].plot(idx, (lower_test_preds[:n_show] + upper_test_preds[:n_show]) / 2, "r-", linewidth=1)
axes[1].scatter(idx, y_test[:n_show], color="red", s=12, zorder=5, alpha=0.7)
axes[1].set_title(f"CQR (asymmetric)\nCoverage={cqr_coverage:.1%}, Avg Width={cqr_widths.mean():.3f}")
axes[1].set_xlabel("Observation")

fig.suptitle("Symmetric vs Asymmetric Conformal Intervals", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## Section 4: Conformal PD Intervals -- ML Uncertainty for Credit Scoring

Conformal prediction naturally extends to classification. For credit scoring, we construct intervals around the predicted probability of default (PD). This bridges Project 03 (Credit Scoring with ML) by adding uncertainty quantification to the point PD estimates.

The nonconformity score is $s_i = |\hat{p}(X_i) - Y_i|$, where $\hat{p}$ is the predicted PD and $Y_i \in \{0, 1\}$ is the true default indicator.

In [ ]:
# Fit conformal PD model
cpd = ConformalPD(
    alpha=alpha,
    n_estimators=config["cqr"]["n_estimators"],
    max_depth=config["cqr"]["max_depth"],
    seed=SEED,
)
cpd.fit(X_cr_train, y_cr_train, X_cr_cal, y_cr_cal)

# Predict intervals on test set
pd_lower, pd_upper = cpd.predict_interval(X_cr_test)
pd_point = cpd._classifier.predict_proba(X_cr_test)[:, 1]
pd_coverage = cpd.coverage(X_cr_test, y_cr_test)

print(f"Conformal PD Coverage: {pd_coverage:.2%} (target: {1-alpha:.0%})")
print(f"PD interval width: mean={np.mean(pd_upper - pd_lower):.4f}, "
      f"median={np.median(pd_upper - pd_lower):.4f}")
print(f"PD bounds: lower in [{pd_lower.min():.4f}, {pd_lower.max():.4f}], "
      f"upper in [{pd_upper.min():.4f}, {pd_upper.max():.4f}]")
print(f"\nTest default rate: {y_cr_test.mean():.2%}")

In [ ]:
# Visualize PD intervals with true default status
fig = plot_conformal_pd(pd_lower, pd_upper, pd_point, y_cr_test, n_show=120)
plt.show()

## Section 5: Adaptive Conformal Inference (ACI) Under Regime Change

Standard conformal prediction assumes exchangeability -- the data distribution is stable. In financial markets, this assumption breaks during regime changes (e.g., a volatility spike). ACI adapts the miscoverage level $\alpha_t$ online:

$$\alpha_{t+1} = \alpha_t + \gamma \cdot (\alpha_{\text{target}} - \text{err}_t)$$

where $\text{err}_t = 1$ if the observation was NOT covered. When coverage drops (more errors), $\alpha_t$ decreases, making the threshold higher (wider intervals). This creates a self-correcting mechanism.

We simulate a regime change: calm period ($\sigma = 0.01$) followed by crisis ($\sigma = 0.04$).

In [ ]:
# Generate regime-change data
n_calm = config["adaptive"]["n_calm"]
n_crisis = config["adaptive"]["n_crisis"]
regime_data = generate_regime_data(n_calm=n_calm, n_crisis=n_crisis, seed=SEED)

print(f"Regime data: {n_calm} calm + {n_crisis} crisis = {len(regime_data)} total")
print(f"Calm volatility:  {np.std(regime_data[:n_calm]):.4f}  (target: 0.01)")
print(f"Crisis volatility: {np.std(regime_data[n_calm:]):.4f}  (target: 0.04)")

# Run ACI experiment
def model_fn(data):
    """Expanding-mean predictor."""
    preds = np.zeros(len(data))
    for i in range(1, len(data)):
        preds[i] = np.mean(data[:i])
    return preds

aci_results = run_aci_experiment(
    data=regime_data,
    model_fn=model_fn,
    alpha=alpha,
    gamma=config["adaptive"]["gamma"],
    cal_size=100,
)
aci_results["n_calm"] = n_calm
aci_results["n_crisis"] = n_crisis

cov_traj = aci_results["coverage_trajectory"]
print(f"\nACI final coverage: {cov_traj[-1]:.2%}")
print(f"ACI coverage at regime change: {cov_traj[n_calm - 100 - 1]:.2%}")
print(f"Number of threshold adjustments: {len(aci_results['thresholds'])}")

In [ ]:
# Visualize ACI results
fig = plot_adaptive_coverage(aci_results, alpha=alpha)
plt.show()

## Section 6: Method Comparison -- Conformal vs Parametric vs Bootstrap

We compare three interval construction methods:
- **Conformal (CQR):** Distribution-free, finite-sample coverage guarantee
- **Parametric (Normal):** Assumes Gaussian returns, uses $\mu \pm z_{\alpha/2} \sigma$
- **Bootstrap:** Resampling-based, asymptotically valid

The comparison uses return-like data to evaluate coverage and interval efficiency (tighter intervals at the same coverage level are preferable).

In [ ]:
# Generate return-like data
returns = rng.standard_normal(config["data"]["n_samples"]) * 0.02

# Use the high-level model
crm = ConformalRiskModel(config)
comparison_df = crm.compare_methods(returns, alpha=alpha)

print("Method Comparison Results:")
print("=" * 55)
print(comparison_df.to_string(index=False, float_format="{:.4f}".format))
print("=" * 55)
print(f"\nTarget coverage: {1-alpha:.0%}")
print("\nKey takeaway: conformal achieves the target coverage without")
print("distributional assumptions, while parametric and bootstrap rely")
print("on normality or large-sample approximations.")

In [ ]:
# Visualize method comparison
fig = plot_interval_width_comparison(comparison_df)
plt.show()

In [ ]:
# Conformal risk control: find the smallest lambda that controls risk
# Risk function: fraction of scores exceeding the threshold lambda
scores_demo = np.abs(rng.standard_normal(500))
lambdas = np.linspace(0.1, 5.0, 100)

def risk_fn(scores, lam):
    return float(np.mean(scores > lam))

lam_star = conformal_risk_control(risk_fn, scores_demo, alpha=0.1, lambdas=lambdas)
risk_at_lam = risk_fn(scores_demo, lam_star)

print(f"Risk control result:")
print(f"  Selected lambda: {lam_star:.4f}")
print(f"  Risk at lambda:  {risk_at_lam:.4f} (target: <= {alpha})")
print(f"  This lambda controls the fraction of exceedances at the {alpha} level.")

## Summary

| Technique | Guarantees | Assumptions | Best for |
|-----------|-----------|-------------|----------|
| Split Conformal | Marginal coverage >= 1-alpha | Exchangeability | Simple, any model |
| CQR | Marginal coverage >= 1-alpha | Exchangeability | Heteroscedastic data |
| ACI | Long-run coverage converges | None (online) | Non-stationary data |
| Conformal Risk Control | Risk <= alpha | Exchangeability | Custom risk functionals |

**Key results from this project:**
- Conformal intervals achieve the target coverage (90%) without distributional assumptions
- CQR produces adaptive-width intervals that are tighter where the model is confident
- ACI successfully adapts to regime changes, widening intervals when volatility spikes
- Conformal PD intervals provide uncertainty quantification for credit scoring models

**Connections to other projects:**
- **P01 (Portfolio Risk):** Conformal VaR intervals complement parametric/historical VaR
- **P03 (Credit Scoring):** Conformal PD adds uncertainty to ML-based default predictions
- **P04 (Volatility):** ACI handles the non-stationarity that GARCH models address parametrically
- **P05 (EVT):** Both address tail risk -- EVT models the tail, conformal wraps any model with guarantees